In [152]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse


***
**Scrapping Function**
***

In [150]:
def extract_text_from_div_p(url, div_class_name):
    if div_class_name=="to-do":
        return "No content available"
    try:
        # Send a GET request to the website
        response = requests.get(url)
        response.raise_for_status()   # Raise an HTTPError for bad responses (4xx and 5xx)

        # Parse the website's HTML content
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the div with the specified class name
        target_div = soup.find('div', class_=div_class_name)
        if target_div:
            # Extract all <p> tags inside the div
            paragraphs = target_div.find_all('p')

            # Join the text from all <p> tags and return it
            return " ".join(p.get_text(strip=True) for p in paragraphs)
        else:
            return f"Div with class '{div_class_name}' not found on the page."

    except requests.exceptions.RequestException as e:
        return f"An error occurred while trying to fetch the website: {e}"

In [151]:
def extract_text_with_selenium(url, div_class_name):
    # Set up Selenium WebDriver options
    options = Options()
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")  # Run in headless mode (no browser UI)
    #options.add_argument("--no-sandbox")  # Bypass OS security model
    #options.add_argument("--disable-dev-shm-usage")  # Prevent resource issues # Replace with the path to your ChromeDriver executable
    driver = webdriver.Chrome(options=options)

    try:
        # Open the webpage
        driver.get(url)
        driver.implicitly_wait(1)
        if div_class_name == "to-do":
            return "No content available"
        if div_class_name == "not-anymore":
            return "website done"
        try:
            target_div = driver.find_element(By.CLASS_NAME, div_class_name)
        except NoSuchElementException:
            return f"Div with class '{div_class_name}' not found on the page."

        # Extract all <p> tags inside the div and join their text
        paragraphs = target_div.find_elements(By.TAG_NAME, 'p')
        return " ".join(p.text for p in paragraphs)

    except Exception as e:
        return f"An error occurred while trying to fetch the website with Selenium: {e}"

    finally:
        # Close the browser
        driver.quit()

***
**Mapping**
***

In [142]:
class_name_mapping_bs = {
    "breizh-info.com" : "elementor-element elementor-element-b1ddf1f contenuarticle elementor-widget elementor-widget-theme-post-content",
    "lalettrepatriote.com":"llp-single-corps",
    "lemediaen442.fr":"inner-content clearµfix",
    #"www.lesakerfrancophone.fr":"entry-content",#403...
    "lesakerfrancophone.fr":"to-do",
    "lesalonbeige.fr":"gp-entry-text",
    "lesdeqodeurs.fr":"elementor-element elementor-element-7d51b89f jltma-glass-effect-no elementor-widget elementor-widget-theme-post-content",
    #"www.lesmoutonsrebelles.com":"", #Existe plus
    "lesmoutonsrebelles.com":"to-do",
    "lesobservateurs.ch":"entry-content",
    "lezarceleurs.blogspot.com":"entry-content",
    #"www.reseauinternational.net":'', #403...
    "reseauinternational.net":'to-do',
    "ripostelaique.com":"entry-content clearfix",
    "bvoltaire.fr":"elementor-element elementor-element-a6916c6 article-content elementor-widget elementor-widget-theme-post-content",
    "dreuz.info":"post-content cf entry-content content-spacious" ,
    "francesoir.fr":"clearfix text-formatted field field--name-body field--type-text-with-summary field--label-hidden field__item",
    "lelibrepenseur.org":"content-inner",
    "polemia.com":"et_pb_module et_pb_post_content et_pb_post_content_0_tb_body",
    "profession-gendarme.com":"entry-content"
}   

In [143]:
class_name_mapping_selenium = {
    "breizh-info.com" : "elementor-element elementor-element-b1ddf1f contenuarticle elementor-widget elementor-widget-theme-post-content",
    "lalettrepatriote.com":"llp-single-corps",
    "lemediaen442.fr":"inner-content clearµfix",
    "lesakerfrancophone.fr":"entry-content",
    "lesalonbeige.fr":"gp-entry-text",
    "lesdeqodeurs.fr":"elementor-element elementor-element-7d51b89f jltma-glass-effect-no elementor-widget elementor-widget-theme-post-content",
    #"www.lesmoutonsrebelles.com":"", #Existe plus
    "lesmoutonsrebelles.com":"not-anymore",
    "lesobservateurs.ch":"entry-content",
    "lezarceleurs.blogspot.com":"entry-content",
    "reseauinternational.net":'post-content-wrap',
    "ripostelaique.com":"entry-content clearfix",
    "bvoltaire.fr":"elementor-element elementor-element-a6916c6 article-content elementor-widget elementor-widget-theme-post-content",
    "dreuz.info":"post-content cf entry-content content-spacious" ,
    "francesoir.fr":"clearfix text-formatted field field--name-body field--type-text-with-summary field--label-hidden field__item",
    "lelibrepenseur.org":"content-inner",
    "polemia.com":"et_pb_module et_pb_post_content et_pb_post_content_0_tb_body",
    "profession-gendarme.com":"entry-content"
}   

***
**Extracting text**
***

In [144]:
dataset = pd.read_csv("obsinfox.csv")
dataset

,URL,Title,Fake News,"Places, Dates, People",Facts,Opinions,Subjective,Reported information,Sources Cited,False Information,Insinuation,Exaggeration,Offbeat Title,Annotator
0,https://lesakerfrancophone.fr/la-relation-entr...,La relation entre la technologie et la religion,0,1,1,1,1,0,1,0,0,0,0,rater1
1,https://www.breizh-info.com/2021/01/27/157958/...,"Confinement. Les habitants de Brest, Morlaix e...",0,1,1,0,0,0,1,0,0,0,0,rater1
2,https://reseauinternational.net/la-chine-le-pr...,La Chine : Le premier marché mondial de Smartp...,0,1,1,0,0,0,1,0,0,0,0,rater1
3,https://lezarceleurs.blogspot.com/2021/12/emma...,"Emmanuel à Olivier : « Tiens bon, on les aura ...",1,1,1,1,1,0,0,1,0,1,0,rater1
4,https://lesakerfrancophone.fr/selon-ubs-les-pr...,"Selon UBS, les « propriétés d’assurance tant d...",0,1,1,1,1,0,1,0,0,0,0,rater1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,https://lemediaen442.fr/loi-climat-le-gouverne...,Loi climat : le gouvernement va exempter Amazo...,0,1,1,0,0,0,0,0,1,1,0,rater8
796,https://www.breizh-info.com/2017/11/10/81481/q...,Le QI peut-il être augmenté par l’éducation ? ...,0,1,1,0,0,0,1,0,0,0,0,rater8
797,https://lesmoutonsrebelles.com/secte-sexuelle-...,Secte sexuelle : l’ex-star de Smallville Aliso...,0,1,1,0,0,0,1,0,1,0,0,rater8
798,https://lesalonbeige.fr/sacre-de-charles-iii-p...,Sacre de Charles III par SAR le Prince Charles...,0,1,1,1,1,0,0,0,1,0,0,rater8


In [ ]:
unique = dataset.drop_duplicates(subset=["URL"])

In [ ]:
#Method 1 :
def get_hostname_and_scrape_bs(row):
    # Extract the hostname from the URL
    hostname = urlparse(row['URL']).netloc
    hostname = urlparse(row['URL']).netloc.replace("www.", "")
    # Find the corresponding div class name from the dictionary
    div_class_name = class_name_mapping_bs[hostname]
    if div_class_name:
        # Use the scraping function with the URL and div class name
        return extract_text_from_div_p(row['URL'], div_class_name)
    else:
        return f"No matching div class for hostname: {hostname}"

# Apply the function to each row in the dataset and store the results in a new column 'text'
unique['text'] = unique.apply(get_hostname_and_scrape_bs, axis=1)

In [ ]:
unique.to_excel('withtext.xlsx', index=False)

In [ ]:
#Method 2
def get_hostname_and_scrape_sel(row):
    # Extract the hostname from the URL
    hostname = urlparse(row['URL']).netloc
    hostname = urlparse(row['URL']).netloc.replace("www.", "")
    # Find the corresponding div class name from the dictionary
    div_class_name = class_name_mapping_selenium[hostname]
    if div_class_name:
        # Use the scraping function with the URL and div class name
        return extract_text_with_selenium(row['URL'], div_class_name)
    else:
        return f"No matching div class for hostname: {hostname}"

# Apply the function to each row in the dataset and store the results in a new column 'text'
unique['text'] = unique.apply(get_hostname_and_scrape_sel, axis=1)


C:\Users\mathys.bodelet\AppData\Local\Temp\ipykernel_15540\2553311440.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unique['text'] = unique.apply(get_hostname_and_scrape, axis=1)


In [148]:
unique.to_excel('withtext_sel.xlsx', index=False)

By comparing results from both file I was able to get the text of all links except for the ones that were not available online anymore.